# **SETUP**

In [22]:
import pathlib
import pandas as pd
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = pathlib.Path().absolute().parent

train = pd.read_parquet(PROJECT_ROOT / "data" / "train.parquet")
test = pd.read_parquet(PROJECT_ROOT / "data" / "test.parquet")
cv = pd.read_parquet(PROJECT_ROOT / "data" / "cv.parquet")

std = StandardScaler().set_output(transform="pandas")

# **STANDARD SCALE NUMERICS**

In [23]:
train_subset = train.select_dtypes("number").drop(columns=["PitNextLap"])
test_subset = test.select_dtypes("number")
oofs = []

for k in sorted(cv.outer_fold.unique()):
    is_val = cv["outer_fold"] == k
    std.fit(train_subset[~is_val])
    oofs.append(std.transform(train_subset[is_val]))

_ = std.fit(train_subset)

# **EXPORT**

In [25]:
feature_name = "003-standard-scale-numerics"
(PROJECT_ROOT / "data" / "features" / feature_name).mkdir(exist_ok=True)

oof_out = pd.concat(oofs).sort_index()
oof_out.to_parquet(PROJECT_ROOT / "data" / "features" / feature_name / "oof.parquet")

test_out = std.transform(test_subset)
test_out.to_parquet(PROJECT_ROOT / "data" / "features" / feature_name / "test.parquet")